In [ ]:
!pip install torch torchvision torchaudio
!pip install pandas numpy matplotlib tqdm opencv-python

!pip install -q opencv-python==4.10.0.84
!pip install -q opencv-contrib-python==4.10.0.84
!pip install -q ultralytics
!pip install -q numpy==1.23.5 --force-reinstall
!pip install -q matplotlib==3.9.0
!pip install -q tqdm==4.66.4

print("\n✓ パッケージのインストールが完了しました")

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive
    
    # Google Driveをマウント
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリに移動
    PROJECT_ROOT = '/content/drive/MyDrive/Visuable_for_you_tabletennis'
    os.chdir(PROJECT_ROOT)
    
    # notebooksディレクトリをパスに追加
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts/notebooks'))
    
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True
    
except ImportError:
    # ローカル環境の場合
    IN_COLAB = False
    # notebooksディレクトリ（このノートブックの場所）をパスに追加
    notebook_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
import torch
import json
from pathlib import Path

# パイプラインのインポート
from src.pipelines import InferencePipeline
from src.pipelines.config import InferencePipelineConfig

print("✓ モジュールのインポートが完了しました")
print(f"PyTorchバージョン: {torch.__version__}")
print(f"CUDAが利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 設定ファイルのパス
CONFIG_PATH = 'configs/crip_app_config.json'

# 設定の読み込み
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, 'r') as f:
        config_dict = json.load(f)
    print(f"✓ 設定ファイルを読み込みました: {CONFIG_PATH}")
else:
    print(f"警告: 設定ファイルが見つかりません: {CONFIG_PATH}")

# 設定の表示
print("\n設定内容:")
for key, value in config_dict.items():
    print(f"  {key}: {value}")

In [ ]:
# ============================================================
# パス設定（ここを変更してください）
# ============================================================

# 入力動画
INPUT_VIDEO = 'data/raw/sample_video_01_02.MOV'

# 出力ディレクトリ
OUTPUT_DIR = 'output/predictions/sample_video_01_02'

# ベース名（Noneの場合は入力動画名を使用）
BASE_NAME = None

# ============================================================

# ファイル存在確認
print("="*60)
print("ファイルの確認")
print("="*60)
print(f"入力動画: {'✓' if os.path.exists(INPUT_VIDEO) else '✗'} {INPUT_VIDEO}")
print(f"卓球台検出モデル: {'✓' if os.path.exists(config_dict['models']['table_detection']) else '✗'} {config_dict['models']['table_detection']}")
print(f"姿勢推定モデル: {'✓' if os.path.exists(config_dict['models']['pose_estimation']) else '✗'} {config_dict['models']['pose_estimation']}")
print(f"プレー分類モデル: {'✓' if os.path.exists(config_dict['models']['play_classifier']) else '✗'} {config_dict['models']['play_classifier']}")
print(f"出力ディレクトリ: {OUTPUT_DIR}")
print("="*60)

In [ ]:
# InferencePipelineConfigを作成
from src.pipelines.config import (
    PlayerPoseExporterConfig,
    TableDetectionConfig,
    PoseTrackingConfig,
    PlayerClassificationConfig,
    TrackingExportConfig,
    VideoProcessingConfig,
    PlaySceneDetectionConfig,
)

# 各設定を作成
table_detection_config = TableDetectionConfig(
    model_path=config_dict['models']['table_detection'],
    device=config_dict.get('device', 'cuda'),
    min_confidence=config_dict.get('table_detection', {}).get('min_confidence', 0.5),
    max_detection_attempts=config_dict.get('table_detection', {}).get('max_detection_attempts', 200)
)

pose_tracking_config = PoseTrackingConfig(
    model_path=config_dict['models']['pose_estimation'],
    device=config_dict.get('device', 'cuda'),
    conf_threshold=config_dict.get('pose_tracking', {}).get('conf_threshold', 0.5),
    iou_threshold=config_dict.get('pose_tracking', {}).get('iou_threshold', 0.7),
    table_distance_threshold=config_dict.get('pose_tracking', {}).get('table_distance_threshold', 0.2),
    min_keypoint_confidence=config_dict.get('pose_tracking', {}).get('min_keypoint_confidence', 0.5)
)

player_classification_config = PlayerClassificationConfig(
    near_table_threshold=config_dict.get('player_classification', {}).get('near_table_threshold', 0.1),
    min_tracking_frames=config_dict.get('player_classification', {}).get('min_tracking_frames', 10),
    max_players=config_dict.get('player_classification', {}).get('max_players', 4),
    max_inactive_frames=config_dict.get('player_classification', {}).get('max_inactive_frames', 30),
    min_player_score=config_dict.get('player_classification', {}).get('min_player_score', 0.3),
    recent_frames_window=config_dict.get('player_classification', {}).get('recent_frames_window', 146),
    max_consecutive_other_count=config_dict.get('player_classification', {}).get('max_consecutive_other_count', 30),
    movement_noise_threshold=config_dict.get('player_classification', {}).get('movement_noise_threshold', 5.0)
)

tracking_export_config = TrackingExportConfig(
    min_consecutive_frames=config_dict.get('tracking_export', {}).get('min_consecutive_frames', 30),
    max_frame_gap=config_dict.get('tracking_export', {}).get('max_frame_gap', 5)
)

video_processing_config = VideoProcessingConfig(
    target_fps=config_dict.get('video_processing', {}).get('target_fps', 30.0),
    show_progress=config_dict.get('video_processing', {}).get('show_progress', True),
    output_codec=config_dict.get('video_processing', {}).get('output_codec', 'mp4v')
)

pose_export_config = PlayerPoseExporterConfig(
    table_detection=table_detection_config,
    pose_tracking=pose_tracking_config,
    player_classification=player_classification_config,
    tracking_export=tracking_export_config,
    video_processing=video_processing_config,
    save_intermediate_files=config_dict.get('pipeline', {}).get('save_intermediate_files', True)
)

scene_detection_config = PlaySceneDetectionConfig(
    model_path=config_dict['models']['play_classifier'],
    config_path=config_dict['models'].get('play_classifier_config'),
    device=config_dict.get('device', 'cuda'),
    threshold=config_dict.get('scene_detection', {}).get('threshold', 0.5),
    min_scene_duration=config_dict.get('scene_detection', {}).get('min_scene_duration', 10)
)

pipeline_config = InferencePipelineConfig(
    pose_export=pose_export_config,
    scene_detection=scene_detection_config,
    show_progress=config_dict.get('pipeline', {}).get('show_progress', True),
    save_intermediate_files=config_dict.get('pipeline', {}).get('save_intermediate_files', True)
)

# パイプラインの初期化
pipeline = InferencePipeline(config=pipeline_config)

print("\n✓ パイプライン初期化完了")

In [ ]:
# パイプライン実行
results = pipeline.process_video(
    input_video=INPUT_VIDEO,
    output_dir=OUTPUT_DIR,
    base_name=BASE_NAME
)

print("\n✓ パイプライン処理が完了しました")

In [ ]:
from IPython.display import Video, display, Image
import matplotlib.pyplot as plt

print("="*70)
print("処理結果サマリー")
print("="*70)
print(f"\n入力動画: {results['input_video']}")
print(f"出力ディレクトリ: {results['output_dir']}")

print(f"\n【骨格データ抽出】")
print(f"  処理フレーム数: {results['pose_export']['processed_frames']}")
print(f"  検出プレイヤー: {results['pose_export']['player_ids']}")
print(f"  出力CSV: {results['output_files']['pose_csv']}")
print(f"  出力動画: {results['output_files']['pose_video']}")

print(f"\n【プレーシーン検出】")
print(f"  検出シーン数: {results['scene_detection']['total_scenes']}")
print(f"  判定閾値: {results['scene_detection']['threshold']}")
print(f"  最小シーン長: {results['scene_detection']['min_scene_duration']}フレーム")

if results['scene_detection']['total_scenes'] > 0:
    print(f"\n  主要シーン（最初の5シーン）:")
    for i, (start, end) in enumerate(results['scene_detection']['scenes'][:5], 1):
        duration = end - start + 1
        print(f"    シーン{i}: フレーム {start}-{end} ({duration}フレーム)")

print(f"\n【ハイライト動画】")
print(f"  総シーン数: {results['video_composition']['total_scenes']}")
print(f"  総フレーム数: {results['video_composition']['total_frames']}")
print(f"  動画時間: {results['video_composition']['duration_sec']:.1f}秒")
if results['output_files']['play_scenes_video']:
    print(f"  出力動画: {results['output_files']['play_scenes_video']}")

print("="*70)

# グラフの表示（Colab環境の場合）
if IN_COLAB:
    output_base = Path(results['output_dir']) / Path(results['input_video']).stem
    graph_path = f"{output_base}_prediction_graph.png"
    
    if os.path.exists(graph_path):
        print(f"\n予測グラフ:")
        display(Image(graph_path))
    
    # ハイライト動画の表示
    if results['output_files']['play_scenes_video'] and os.path.exists(results['output_files']['play_scenes_video']):
        print(f"\nハイライト動画のプレビュー:")
        display(Video(results['output_files']['play_scenes_video'], width=800))